In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🔐 API Authentication & Authorization Guide

**Implementing secure API authentication and fine-grained authorization controls**

This guide covers:
- Authentication methods (JWT, OAuth2, API keys)
- Token management and revocation
- Authorization and access control
- Role-based access control (RBAC)
- Attribute-based access control (ABAC)
- Scopes and permissions
- API key security
- Audit and compliance
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Authentication Methods

### JWT Token Authentication
```python
from dataclasses import dataclass, field
from typing import Dict, Optional, List
from datetime import datetime, timedelta
import hashlib
import json
import base64

@dataclass
class JWTClaims:
    """JWT token claims"""
    sub: str  # Subject (user ID)
    iss: str  # Issuer
    aud: str  # Audience
    exp: int  # Expiration timestamp
    iat: int  # Issued at
    jti: str  # JWT ID (unique identifier)
    scopes: List[str] = field(default_factory=list)
    custom_claims: Dict = field(default_factory=dict)

class JWTTokenManager:
    """Manage JWT tokens"""
    
    def __init__(self, secret_key: str, issuer: str):
        self.secret_key = secret_key
        self.issuer = issuer
        self.revoked_tokens: set = set()
        self.token_whitelist: Dict[str, dict] = {}
    
    def generate_token(self, user_id: str, scopes: List[str], expires_in_hours: int = 24) -> str:
        """Generate JWT token"""
        
        now = datetime.utcnow()
        expires_at = now + timedelta(hours=expires_in_hours)
        
        claims = JWTClaims(
            sub=user_id,
            iss=self.issuer,
            aud="api",
            exp=int(expires_at.timestamp()),
            iat=int(now.timestamp()),
            jti=hashlib.sha256(f"{user_id}{now.timestamp()}".encode()).hexdigest()[:16],
            scopes=scopes
        )
        
        # Encode header
        header = {"alg": "HS256", "typ": "JWT"}
        header_encoded = base64.urlsafe_b64encode(json.dumps(header).encode()).decode().rstrip('=')
        
        # Encode payload
        payload = {
            'sub': claims.sub,
            'iss': claims.iss,
            'aud': claims.aud,
            'exp': claims.exp,
            'iat': claims.iat,
            'jti': claims.jti,
            'scopes': claims.scopes
        }
        payload_encoded = base64.urlsafe_b64encode(json.dumps(payload).encode()).decode().rstrip('=')
        
        # Create signature
        message = f"{header_encoded}.{payload_encoded}"
        signature = hashlib.sha256(f"{message}{self.secret_key}".encode()).hexdigest()
        signature_encoded = base64.urlsafe_b64encode(signature.encode()).decode().rstrip('=')
        
        token = f"{message}.{signature_encoded}"
        
        # Store in whitelist
        self.token_whitelist[claims.jti] = {
            'user_id': user_id,
            'scopes': scopes,
            'expires_at': expires_at.isoformat()
        }
        
        return token
    
    def verify_token(self, token: str) -> Optional[JWTClaims]:
        """Verify JWT token"""
        
        try:
            parts = token.split('.')
            
            if len(parts) != 3:
                return None
            
            header_encoded, payload_encoded, signature_encoded = parts
            
            # Verify signature
            message = f"{header_encoded}.{payload_encoded}"
            expected_signature = hashlib.sha256(f"{message}{self.secret_key}".encode()).hexdigest()
            
            if not self._verify_signature(expected_signature, signature_encoded):
                return None
            
            # Decode payload
            padding = (4 - len(payload_encoded) % 4) % 4
            payload_encoded += '=' * padding
            
            payload = json.loads(base64.urlsafe_b64decode(payload_encoded))
            
            # Check expiration
            if payload['exp'] < datetime.utcnow().timestamp():
                return None
            
            # Check if revoked
            if payload['jti'] in self.revoked_tokens:
                return None
            
            # Check if token was issued by this manager
            if payload['jti'] not in self.token_whitelist:
                return None
            
            return JWTClaims(
                sub=payload['sub'],
                iss=payload['iss'],
                aud=payload['aud'],
                exp=payload['exp'],
                iat=payload['iat'],
                jti=payload['jti'],
                scopes=payload.get('scopes', [])
            )
        
        except Exception as e:
            print(f"Token verification failed: {e}")
            return None
    
    def _verify_signature(self, expected: str, provided: str) -> bool:
        """Verify signature"""
        
        # Simple comparison (in production, use constant-time comparison)
        padding = (4 - len(provided) % 4) % 4
        provided_padded = provided + '=' * padding
        
        try:
            provided_bytes = base64.urlsafe_b64decode(provided_padded)
            return provided_bytes.decode() == expected
        except Exception:
            return False
    
    def revoke_token(self, jti: str):
        """Revoke token"""
        self.revoked_tokens.add(jti)
```

### OAuth2 Authorization Flow
```python
@dataclass
class OAuthClient:
    """OAuth2 client"""
    client_id: str
    client_secret: str
    redirect_uris: List[str]
    allowed_scopes: List[str]
    grant_types: List[str]

class OAuth2AuthorizationServer:
    """OAuth2 authorization server"""
    
    def __init__(self):
        self.clients: Dict[str, OAuthClient] = {}
        self.authorization_codes: Dict[str, Dict] = {}
        self.access_tokens: Dict[str, Dict] = {}
    
    def register_client(self, client: OAuthClient) -> bool:
        """Register OAuth2 client"""
        
        self.clients[client.client_id] = client
        return True
    
    def generate_authorization_code(self, client_id: str, user_id: str, 
                                   requested_scopes: List[str]) -> Optional[str]:
        """Generate authorization code"""
        
        if client_id not in self.clients:
            return None
        
        client = self.clients[client_id]
        
        # Validate scopes
        if not all(scope in client.allowed_scopes for scope in requested_scopes):
            return None
        
        code = hashlib.sha256(f"{client_id}{user_id}{datetime.now().timestamp()}".encode()).hexdigest()[:32]
        
        self.authorization_codes[code] = {
            'client_id': client_id,
            'user_id': user_id,
            'scopes': requested_scopes,
            'created_at': datetime.now(),
            'expires_at': datetime.now() + timedelta(minutes=10)
        }
        
        return code
    
    def exchange_code_for_token(self, code: str, client_id: str, client_secret: str) -> Optional[str]:
        """Exchange authorization code for access token"""
        
        if code not in self.authorization_codes:
            return None
        
        auth_code_data = self.authorization_codes[code]
        
        # Verify client
        if auth_code_data['client_id'] != client_id:
            return None
        
        if client_id not in self.clients:
            return None
        
        client = self.clients[client_id]
        
        if client.client_secret != client_secret:
            return None
        
        # Check expiration
        if auth_code_data['expires_at'] < datetime.now():
            return None
        
        # Generate access token
        token = hashlib.sha256(f"{client_id}{auth_code_data['user_id']}{datetime.now().timestamp()}".encode()).hexdigest()
        
        self.access_tokens[token] = {
            'client_id': client_id,
            'user_id': auth_code_data['user_id'],
            'scopes': auth_code_data['scopes'],
            'created_at': datetime.now(),
            'expires_at': datetime.now() + timedelta(hours=24)
        }
        
        # Invalidate code
        del self.authorization_codes[code]
        
        return token
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Authorization & Access Control

### Role-Based Access Control (RBAC)
```python
@dataclass
class Role:
    """Authorization role"""
    role_id: str
    name: str
    permissions: List[str]
    description: str

class RBACEngine:
    """Role-based access control"""
    
    def __init__(self):
        self.roles: Dict[str, Role] = {}
        self.user_roles: Dict[str, List[str]] = {}
    
    def create_role(self, role: Role) -> bool:
        """Create role"""
        self.roles[role.role_id] = role
        return True
    
    def assign_role(self, user_id: str, role_id: str) -> bool:
        """Assign role to user"""
        
        if role_id not in self.roles:
            return False
        
        if user_id not in self.user_roles:
            self.user_roles[user_id] = []
        
        if role_id not in self.user_roles[user_id]:
            self.user_roles[user_id].append(role_id)
        
        return True
    
    def has_permission(self, user_id: str, permission: str) -> bool:
        """Check if user has permission"""
        
        if user_id not in self.user_roles:
            return False
        
        for role_id in self.user_roles[user_id]:
            if role_id in self.roles:
                role = self.roles[role_id]
                if permission in role.permissions:
                    return True
        
        return False
    
    def get_user_permissions(self, user_id: str) -> set:
        """Get all permissions for user"""
        
        permissions = set()
        
        if user_id not in self.user_roles:
            return permissions
        
        for role_id in self.user_roles[user_id]:
            if role_id in self.roles:
                permissions.update(self.roles[role_id].permissions)
        
        return permissions

class ABACEngine:
    """Attribute-based access control"""
    
    def __init__(self):
        self.policies: List[Dict] = []
    
    def add_policy(self, policy: Dict):
        """Add access control policy"""
        self.policies.append(policy)
    
    def evaluate(self, subject: Dict, action: str, resource: Dict) -> bool:
        """Evaluate access based on attributes"""
        
        for policy in self.policies:
            if self._match_policy(policy, subject, action, resource):
                return policy.get('effect') == 'allow'
        
        return False
    
    def _match_policy(self, policy: Dict, subject: Dict, action: str, resource: Dict) -> bool:
        """Check if policy applies"""
        
        # Check subject attributes
        subject_conditions = policy.get('subject', {})
        for attr, value in subject_conditions.items():
            if subject.get(attr) != value:
                return False
        
        # Check action
        if action not in policy.get('actions', []):
            return False
        
        # Check resource attributes
        resource_conditions = policy.get('resource', {})
        for attr, value in resource_conditions.items():
            if resource.get(attr) != value:
                return False
        
        return True
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. API Key Management

### Secure API Keys
```python
class APIKeyManager:
    """Manage API keys securely"""
    
    def __init__(self):
        self.api_keys: Dict[str, Dict] = {}
        self.key_usage: Dict[str, List[datetime]] = {}
    
    def generate_api_key(self, application_id: str, name: str) -> Optional[str]:
        """Generate API key"""
        
        key = hashlib.sha256(f"{application_id}{name}{datetime.now().timestamp()}".encode()).hexdigest()
        key_hash = hashlib.sha256(key.encode()).hexdigest()
        
        self.api_keys[key_hash] = {
            'application_id': application_id,
            'name': name,
            'created_at': datetime.now(),
            'last_used': None,
            'is_active': True,
            'rate_limit_per_minute': 100
        }
        
        return key  # Return only once to user
    
    def verify_api_key(self, key: str) -> Optional[Dict]:
        """Verify API key"""
        
        key_hash = hashlib.sha256(key.encode()).hexdigest()
        
        if key_hash not in self.api_keys:
            return None
        
        key_data = self.api_keys[key_hash]
        
        if not key_data['is_active']:
            return None
        
        # Update usage
        self.key_usage.setdefault(key_hash, []).append(datetime.now())
        key_data['last_used'] = datetime.now()
        
        return key_data
    
    def revoke_api_key(self, key: str) -> bool:
        """Revoke API key"""
        
        key_hash = hashlib.sha256(key.encode()).hexdigest()
        
        if key_hash not in self.api_keys:
            return False
        
        self.api_keys[key_hash]['is_active'] = False
        
        return True
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Authentication & Authorization Checklist

✅ **Authentication**
- [ ] Authentication method chosen
- [ ] JWT implementation
- [ ] Token generation working
- [ ] Token verification
- [ ] Token refresh mechanism

✅ **Authorization**
- [ ] RBAC implemented
- [ ] Roles defined
- [ ] Permissions clear
- [ ] User assignments
- [ ] Access checks enforced

✅ **Security**
- [ ] Secrets encrypted
- [ ] Tokens secured (HTTPS)
- [ ] Revocation working
- [ ] Rate limiting
- [ ] Audit logging

✅ **API Keys**
- [ ] Key generation secure
- [ ] Storage hashed
- [ ] Rotation strategy
- [ ] Revocation tested
- [ ] Usage monitoring

✅ **Operational**
- [ ] Documentation
- [ ] Integration tested
- [ ] Performance verified
- [ ] Monitoring alerts
- [ ] Incident procedures
</VSCode.Cell>
```